# Mini-Courier Planner - Phase 1 and Phase 2

This notebook implements the first two phases of the Mini-Courier Planner capstone assignment.

- **Phase 1:** Courier problem design and planning.
- **Phase 2:** Package prioritization and box filling using sorting and dynamic programming.

The aim is to build the foundation for an intelligent courier planning system that can later be extended with shortest-path and route-planning algorithms.

## Phase 1: Problem Statement and Objective

A courier company receives multiple parcels that must be delivered to different customer locations. Each parcel has a destination, weight, delivery deadline, priority level, and delivery value. Since a delivery vehicle has limited carrying capacity, the system must decide which parcels should be loaded for one trip.

**Objective:**

Design a prototype that can prioritize parcels, choose the best parcels for one delivery trip, and prepare the project structure for future route optimization.

**Current notebook scope:**

- Define parcel attributes and sample parcel data.
- Sort parcels by priority and deadline.
- Use 0/1 knapsack dynamic programming to select the optimal set of parcels for one trip.
- Describe how graph algorithms and route planning will be added in later phases.

## Parcel Data Design

Each parcel is represented using these attributes:

| Attribute | Meaning |
|---|---|
| `parcel_id` | Unique parcel identifier |
| `destination` | Delivery location code |
| `weight` | Parcel weight in kg |
| `value` | Business value or delivery reward |
| `deadline` | Required delivery day; smaller means more urgent |
| `priority` | Priority category: High, Medium, or Low |
| `priority_score` | Numeric score used by algorithms |
| `benefit` | Combined score optimized by knapsack |

For Phase 2, parcel `weight` is the capacity cost, and `benefit` is the optimization value.

## Delivery Point and Graph Design for Later Phases

Although route computation is not implemented in this notebook, the planned graph model is:

- Each delivery point is a graph node.
- Roads between delivery points are weighted edges.
- Edge weight represents travel distance or travel time.
- Dijkstra's algorithm will calculate shortest paths between locations.
- A nearest-neighbor TSP approximation will order the selected delivery points into a practical delivery route.

This means Phase 2 produces the selected parcels, and later phases can use their destinations as input for graph-based route planning.

## Algorithm Plan

| Component | Algorithm | Purpose |
|---|---|---|
| Parcel prioritization | Stable sorting | Order parcels by priority and urgency |
| Box filling | 0/1 knapsack dynamic programming | Select the best set of parcels within vehicle capacity |
| Shortest paths | Dijkstra's algorithm | Find shortest road paths between locations in later phases |
| Route planning | Nearest-neighbor TSP approximation | Build a practical full delivery route in later phases |

Python's built-in `sorted` function is stable, so parcels with the same priority and deadline keep their original input order. This is useful when the original order represents booking sequence or arrival order.

## Phase 2: Package Prioritization and Box Filling

The 0/1 knapsack recurrence used here is:

`dp[i][w] = max(dp[i - 1][w], benefit[i - 1] + dp[i - 1][w - weight[i - 1]])`

where:

- `i` is the number of parcels considered.
- `w` is the available vehicle capacity.
- `dp[i][w]` stores the best achievable benefit.

Time complexity is `O(n * capacity)` and space complexity is `O(n * capacity)` for the DP table.

In [1]:
from dataclasses import dataclass
from typing import List, Tuple


@dataclass(frozen=True)
class Parcel:
    parcel_id: str
    destination: str
    weight: int
    value: int
    deadline: int
    priority: str


PRIORITY_SCORE = {
    "High": 3,
    "Medium": 2,
    "Low": 1,
}


def priority_score(parcel: Parcel) -> int:
    return PRIORITY_SCORE[parcel.priority]


def parcel_benefit(parcel: Parcel) -> int:
    urgency_bonus = max(0, 6 - parcel.deadline) * 5
    priority_bonus = priority_score(parcel) * 20
    return parcel.value + priority_bonus + urgency_bonus


parcels = [
    Parcel("P01", "A", 4, 60, 2, "High"),
    Parcel("P02", "B", 2, 35, 1, "High"),
    Parcel("P03", "C", 6, 80, 4, "Medium"),
    Parcel("P04", "D", 3, 45, 3, "High"),
    Parcel("P05", "E", 5, 70, 2, "Medium"),
    Parcel("P06", "F", 1, 20, 1, "Low"),
    Parcel("P07", "G", 7, 100, 5, "High"),
    Parcel("P08", "H", 2, 25, 3, "Low"),
    Parcel("P09", "I", 2, 30, 3, "High"),
]

VEHICLE_CAPACITY = 12

In [2]:
def sort_parcels_by_priority_and_deadline(parcel_list: List[Parcel]) -> List[Parcel]:
    return sorted(
        parcel_list,
        key=lambda parcel: (-priority_score(parcel), parcel.deadline),
    )


sorted_parcels = sort_parcels_by_priority_and_deadline(parcels)

print("Sorted parcels by priority and deadline:")
for parcel in sorted_parcels:
    print(
        f"{parcel.parcel_id} | destination={parcel.destination} | "
        f"priority={parcel.priority:<6} | deadline={parcel.deadline} | "
        f"weight={parcel.weight} | benefit={parcel_benefit(parcel)}"
    )

Sorted parcels by priority and deadline:
P02 | destination=B | priority=High   | deadline=1 | weight=2 | benefit=120
P01 | destination=A | priority=High   | deadline=2 | weight=4 | benefit=140
P04 | destination=D | priority=High   | deadline=3 | weight=3 | benefit=120
P09 | destination=I | priority=High   | deadline=3 | weight=2 | benefit=105
P07 | destination=G | priority=High   | deadline=5 | weight=7 | benefit=165
P05 | destination=E | priority=Medium | deadline=2 | weight=5 | benefit=130
P03 | destination=C | priority=Medium | deadline=4 | weight=6 | benefit=130
P06 | destination=F | priority=Low    | deadline=1 | weight=1 | benefit=65
P08 | destination=H | priority=Low    | deadline=3 | weight=2 | benefit=60


In [3]:
def knapsack_select_parcels(
    parcel_list: List[Parcel], capacity: int
) -> Tuple[int, List[Parcel], List[List[int]]]:
    n = len(parcel_list)
    dp = [[0 for _ in range(capacity + 1)] for _ in range(n + 1)]

    for i in range(1, n + 1):
        parcel = parcel_list[i - 1]
        weight = parcel.weight
        benefit = parcel_benefit(parcel)

        for current_capacity in range(capacity + 1):
            without_parcel = dp[i - 1][current_capacity]
            with_parcel = 0

            if weight <= current_capacity:
                with_parcel = benefit + dp[i - 1][current_capacity - weight]

            dp[i][current_capacity] = max(without_parcel, with_parcel)

    selected = []
    remaining_capacity = capacity

    for i in range(n, 0, -1):
        if dp[i][remaining_capacity] != dp[i - 1][remaining_capacity]:
            parcel = parcel_list[i - 1]
            selected.append(parcel)
            remaining_capacity -= parcel.weight

    selected.reverse()
    return dp[n][capacity], selected, dp


maximum_benefit, selected_parcels, dp_table = knapsack_select_parcels(
    sorted_parcels,
    VEHICLE_CAPACITY,
)

total_weight = sum(parcel.weight for parcel in selected_parcels)
total_benefit = sum(parcel_benefit(parcel) for parcel in selected_parcels)
remaining_capacity = VEHICLE_CAPACITY - total_weight

print(f"Vehicle capacity: {VEHICLE_CAPACITY} kg")
print(f"Maximum benefit from DP table: {maximum_benefit}")
print("Selected parcels:", ", ".join(parcel.parcel_id for parcel in selected_parcels))
print(f"Total selected weight: {total_weight} kg")
print(f"Total selected benefit: {total_benefit}")
print(f"Remaining capacity: {remaining_capacity} kg")

Vehicle capacity: 12 kg
Maximum benefit from DP table: 550
Selected parcels: P02, P01, P04, P09, P06
Total selected weight: 12 kg
Total selected benefit: 550
Remaining capacity: 0 kg


In [4]:
print(f"DP table rows: {len(dp_table)}")
print(f"DP table columns: {len(dp_table[0])}")
print("Last DP row:", dp_table[-1])

DP table rows: 10
DP table columns: 13
Last DP row: [0, 65, 120, 185, 225, 290, 305, 350, 410, 430, 470, 490, 550]


## Test Cases and Verification

The following tests verify the normal case, a small-capacity edge case, and stable sorting behavior for parcels with the same priority and deadline.

In [5]:
# Test 1: Normal case
assert total_weight <= VEHICLE_CAPACITY
assert maximum_benefit == total_benefit == 550
print("Normal case passed: selected parcels fit capacity and reach optimum benefit.")

# Test 2: Capacity edge case
small_capacity_benefit, small_capacity_selection, _ = knapsack_select_parcels(
    sorted_parcels,
    capacity=3,
)
assert small_capacity_benefit == 185
assert sum(parcel.weight for parcel in small_capacity_selection) <= 3
print("Small capacity edge case passed.")

# Test 3: Stable sorting tie case
sorted_ids = [parcel.parcel_id for parcel in sorted_parcels]
assert sorted_ids.index("P04") < sorted_ids.index("P09")
print("Stable sorting tie case passed: P04 remains before P09.")

Normal case passed: selected parcels fit capacity and reach optimum benefit.
Small capacity edge case passed.
Stable sorting tie case passed: P04 remains before P09.


## Phase 1 and Phase 2 Conclusion

This notebook completes the planning and package-selection foundation for the Mini-Courier Planner.

- Phase 1 defines the problem, parcel structure, graph structure, and algorithm flow.
- Phase 2 sorts parcels by priority and deadline.
- Phase 2 uses dynamic programming to choose the best parcel set within vehicle capacity.
- The selected parcel destinations can be passed to Dijkstra's algorithm and nearest-neighbor route planning in the next phase.